# Exploratory Data Analysis - Visa Approval Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create reports directory
os.makedirs('../reports/figures', exist_ok=True)

print('Libraries imported successfully')

In [ ]:
# Load data
data_path = '../data/raw/h1b16.csv'
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print(f'\nDataset info:')
df.info()

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Basic statistics
print('Numerical columns statistics:')
df.describe()

In [ ]:
# Missing values analysis
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print('Missing Values Analysis:')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Case status distribution
if 'CASE_STATUS' in df.columns:
    plt.figure(figsize=(10, 6))
    case_status_counts = df['CASE_STATUS'].value_counts()
    sns.barplot(x=case_status_counts.index, y=case_status_counts.values)
    plt.title('Case Status Distribution')
    plt.xlabel('Case Status')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('../reports/figures/case_status_distribution.png', dpi=300)
    plt.show()
    
    print('Case Status Counts:')
    print(case_status_counts)

In [ ]:
# Wage distribution
if 'PREVAILING_WAGE' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Original wage distribution
    sns.histplot(df['PREVAILING_WAGE'], bins=50, ax=axes[0])
    axes[0].set_title('Prevailing Wage Distribution')
    axes[0].set_xlabel('Wage')
    axes[0].set_ylabel('Count')
    
    # Log-transformed wage distribution
    sns.histplot(np.log1p(df['PREVAILING_WAGE']), bins=50, ax=axes[1])
    axes[1].set_title('Log-Transformed Wage Distribution')
    axes[1].set_xlabel('Log(Wage + 1)')
    axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/wage_distribution.png', dpi=300)
    plt.show()

In [ ]:
# Year-wise applications
if 'YEAR' in df.columns:
    plt.figure(figsize=(10, 6))
    year_counts = df['YEAR'].value_counts().sort_index()
    sns.barplot(x=year_counts.index, y=year_counts.values)
    plt.title('Applications by Year')
    plt.xlabel('Year')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('../reports/figures/applications_by_year.png', dpi=300)
    plt.show()
    
    print('Applications by Year:')
    print(year_counts)

In [ ]:
# Top employers
if 'EMPLOYER_NAME' in df.columns:
    top_employers = df['EMPLOYER_NAME'].value_counts().head(20)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_employers.values, y=top_employers.index)
    plt.title('Top 20 Employers')
    plt.xlabel('Count')
    plt.ylabel('Employer Name')
    plt.tight_layout()
    plt.savefig('../reports/figures/top_employers.png', dpi=300)
    plt.show()
    
    print('Top 20 Employers:')
    print(top_employers)

In [ ]:
# Top job titles
if 'JOB_TITLE' in df.columns:
    top_jobs = df['JOB_TITLE'].value_counts().head(20)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_jobs.values, y=top_jobs.index)
    plt.title('Top 20 Job Titles')
    plt.xlabel('Count')
    plt.ylabel('Job Title')
    plt.tight_layout()
    plt.savefig('../reports/figures/top_job_titles.png', dpi=300)
    plt.show()
    
    print('Top 20 Job Titles:')
    print(top_jobs)

In [ ]:
# Full-time position distribution
if 'FULL_TIME_POSITION' in df.columns:
    plt.figure(figsize=(8, 6))
    ft_counts = df['FULL_TIME_POSITION'].value_counts()
    sns.barplot(x=ft_counts.index, y=ft_counts.values)
    plt.title('Full-Time Position Distribution')
    plt.xlabel('Full-Time Position')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('../reports/figures/full_time_distribution.png', dpi=300)
    plt.show()
    
    print('Full-Time Position Counts:')
    print(ft_counts)

In [ ]:
# Works location analysis
if 'WORKSITE' in df.columns:
    # Extract state
    def extract_state(worksite):
        if pd.isna(worksite):
            return 'Unknown'
        try:
            parts = worksite.split(',')
            if len(parts) >= 2:
                return parts[-1].strip().upper()
            return 'Unknown'
        except:
            return 'Unknown'
    
    df['STATE'] = df['WORKSITE'].apply(extract_state)
    
    top_states = df['STATE'].value_counts().head(20)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_states.values, y=top_states.index)
    plt.title('Top 20 States by Application Count')
    plt.xlabel('Count')
    plt.ylabel('State')
    plt.tight_layout()
    plt.savefig('../reports/figures/top_states.png', dpi=300)
    plt.show()
    
    print('Top 20 States:')
    print(top_states)

In [ ]:
# Certified vs Denied comparison (for binary classification)
if 'CASE_STATUS' in df.columns:
    binary_df = df[df['CASE_STATUS'].isin(['CERTIFIED', 'DENIED'])]
    
    if len(binary_df) > 0:
        plt.figure(figsize=(10, 6))
        binary_status_counts = binary_df['CASE_STATUS'].value_counts()
        sns.barplot(x=binary_status_counts.index, y=binary_status_counts.values)
        plt.title('Certified vs Denied (Binary Classification)')
        plt.xlabel('Case Status')
        plt.ylabel('Count')
        plt.tight_layout()
        plt.savefig('../reports/figures/certified_vs_denied.png', dpi=300)
        plt.show()
        
        print('Certified vs Denied Counts:')
        print(binary_status_counts)
        print(f'\nCertification Rate: {binary_status_counts.get("CERTIFIED", 0) / len(binary_df) * 100:.2f}%')

In [ ]:
# Summary statistics
print('\n' + '='*50)
print('EDA SUMMARY')
print('='*50)
print(f'Total Records: {len(df)}')
print(f'Total Columns: {len(df.columns)}')
print(f'Missing Values: {df.isnull().sum().sum()}')
print(f'Duplicate Records: {df.duplicated().sum()}')
print(f'\nCase Statuses: {df["CASE_STATUS"].nunique()} unique values')
print(f'Employers: {df["EMPLOYER_NAME"].nunique()} unique values')
print(f'Job Titles: {df["JOB_TITLE"].nunique()} unique values')
print(f'\nAverage Prevailing Wage: ${df["PREVAILING_WAGE"].mean():,.2f}')
print(f'Median Prevailing Wage: ${df["PREVAILING_WAGE"].median():,.2f}')
print('='*50)